# Multiclass Classifier Seed07 Workflow

This notebook reuses the single `seed07` multiclass classifier checkpoint from the full labeled `all_80_10_10` workflow and can also review the unlabeled pseudo-label outputs that feed the UMAP follow-up work.

Recommended order:
- confirm the run produced a `best_model.pt`, `metrics.json`, and `test_predictions.csv`
- review held-out test results for `seed07`
- inspect the saved unlabeled pseudo-label CSVs and summary JSON
- recreate the UMAP plots directly from the saved UMAP coordinate CSV

The UMAP section below reads the exported CSV directly, so it can run even if you only want plotting and not classifier re-inference.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

NOTEBOOK_DIR = REPO_ROOT / "experiments/classifier/multiclass/x64/umap"
SEED07_DIR = NOTEBOOK_DIR.parent / "seed07"

from wafer_defect.config import load_toml

In [ ]:
data_config_path = SEED07_DIR / "data_config.toml"
train_config_path = SEED07_DIR / "train_config.toml"
data_config = load_toml(data_config_path)
train_config = load_toml(train_config_path)

seed07_artifact_dir = SEED07_DIR / "artifacts" / Path(train_config["training"]["output_dir"]).name
checkpoint_path = seed07_artifact_dir / "best_model.pt"
history_path = seed07_artifact_dir / "history.csv"
metrics_path = seed07_artifact_dir / "metrics.json"
test_predictions_path = seed07_artifact_dir / "test_predictions.csv"
unlabeled_summary_path = seed07_artifact_dir / "unlabeled_predictions.summary.json"
unlabeled_predictions_raw_path = seed07_artifact_dir / "unlabeled_predictions.seed07.raw.csv"
unlabeled_predictions_accepted_path = seed07_artifact_dir / "unlabeled_predictions.seed07.accepted.csv"

required_paths = [checkpoint_path, metrics_path, test_predictions_path]
missing_paths = [path for path in required_paths if not path.exists()]
has_required_seed07_artifacts = not missing_paths

print("Data config:", data_config_path)
print("Seed07 config:", train_config_path)
print("Seed07 artifact dir:", seed07_artifact_dir)
print("Checkpoint:", checkpoint_path)
print("Test predictions path:", test_predictions_path)
print("Unlabeled raw path:", unlabeled_predictions_raw_path)
print("Unlabeled accepted path:", unlabeled_predictions_accepted_path)
print("Unlabeled summary path:", unlabeled_summary_path)

if missing_paths:
    print("Missing required seed07 classifier artifacts:")
    for path in missing_paths:
        print(" -", path)

In [ ]:
history = None
metrics = None
test_predictions = None
unlabeled_predictions_raw = None
unlabeled_predictions_accepted = None
unlabeled_summary = None

if has_required_seed07_artifacts:
    history = pd.read_csv(history_path) if history_path.exists() else None
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    test_predictions = pd.read_csv(test_predictions_path)
    if unlabeled_predictions_raw_path.exists():
        unlabeled_predictions_raw = pd.read_csv(unlabeled_predictions_raw_path)
    if unlabeled_predictions_accepted_path.exists():
        unlabeled_predictions_accepted = pd.read_csv(unlabeled_predictions_accepted_path)
    if unlabeled_summary_path.exists():
        unlabeled_summary = json.loads(unlabeled_summary_path.read_text(encoding="utf-8"))

    print("Test accuracy:", metrics["test"]["accuracy"])
    print("Test balanced accuracy:", metrics["test"]["balanced_accuracy"])
    if history is not None:
        print("Best validation balanced accuracy:", history["val_balanced_accuracy"].max())
    if unlabeled_predictions_raw is not None and "accepted_for_pseudo_label" in unlabeled_predictions_raw.columns:
        accepted_fraction = unlabeled_predictions_raw["accepted_for_pseudo_label"].astype(str).str.lower().map({"true": True, "false": False}).mean()
        print("Accepted pseudo-label fraction:", accepted_fraction)
    elif unlabeled_predictions_accepted is not None and unlabeled_predictions_raw is not None:
        print("Accepted pseudo-label fraction:", len(unlabeled_predictions_accepted) / len(unlabeled_predictions_raw))

    display(test_predictions.head())
    if unlabeled_predictions_raw is not None:
        display(unlabeled_predictions_raw.head())
    if unlabeled_predictions_accepted is not None:
        display(unlabeled_predictions_accepted.head())
    if unlabeled_summary is not None:
        display(pd.json_normalize(unlabeled_summary, sep="."))
else:
    print("Skipping seed07 artifact review because required files are missing. The UMAP CSV plotting cells below can still run independently.")

In [ ]:
if unlabeled_predictions_raw is not None:
    raw_label_col = "pseudo_label" if "pseudo_label" in unlabeled_predictions_raw.columns else None
    accepted_flag_col = "accepted_for_pseudo_label" if "accepted_for_pseudo_label" in unlabeled_predictions_raw.columns else None

    if raw_label_col is None:
        print("Raw pseudo-label CSV does not contain a pseudo_label column.")
    else:
        raw_counts = (
            unlabeled_predictions_raw[raw_label_col]
            .fillna("<missing>")
            .value_counts(dropna=False)
            .rename_axis("pseudo_label")
            .rename("raw_count")
            .to_frame()
        )

        if accepted_flag_col is not None:
            accepted_mask = unlabeled_predictions_raw[accepted_flag_col].astype(str).str.lower().eq("true")
            accepted_counts = (
                unlabeled_predictions_raw.loc[accepted_mask, raw_label_col]
                .fillna("<missing>")
                .value_counts(dropna=False)
                .rename_axis("pseudo_label")
                .rename("accepted_count")
                .to_frame()
            )
        elif unlabeled_predictions_accepted is not None and raw_label_col in unlabeled_predictions_accepted.columns:
            accepted_counts = (
                unlabeled_predictions_accepted[raw_label_col]
                .fillna("<missing>")
                .value_counts(dropna=False)
                .rename_axis("pseudo_label")
                .rename("accepted_count")
                .to_frame()
            )
        else:
            accepted_counts = pd.DataFrame(columns=["accepted_count"])

        pseudo_label_summary = raw_counts.join(accepted_counts, how="outer").fillna(0)
        pseudo_label_summary[["raw_count", "accepted_count"]] = pseudo_label_summary[["raw_count", "accepted_count"]].astype(int)
        pseudo_label_summary["accepted_rate"] = (
            pseudo_label_summary["accepted_count"]
            .div(pseudo_label_summary["raw_count"].where(pseudo_label_summary["raw_count"] > 0))
            .fillna(0.0)
        )

        display(pseudo_label_summary.sort_values(["accepted_count", "raw_count"], ascending=[False, False]))
else:
    print("No raw pseudo-label CSV found, so class-level pseudo-label counts are unavailable.")

## Generate UMAP Plots From Saved CSV

These cells load the saved coordinate CSV directly and recreate the three `10a_style` plots without recomputing embeddings.

In [ ]:
UMAP_POINTS_CSV = NOTEBOOK_DIR / "upload_artifacts" / "umap_10a_style" / "embedding_umap_points_10a_style.csv"
UMAP_PLOTS_DIR = NOTEBOOK_DIR / "upload_artifacts" / "umap_10a_style"

umap_df = pd.read_csv(UMAP_POINTS_CSV)
umap_df["score"] = pd.to_numeric(umap_df.get("score", pd.Series(index=umap_df.index, dtype=float)), errors="coerce")
umap_df["pseudo_label_confidence"] = pd.to_numeric(
    umap_df.get("pseudo_label_confidence", pd.Series(index=umap_df.index, dtype=float)),
    errors="coerce",
)
accepted_series = umap_df.get("accepted_for_pseudo_label", pd.Series(index=umap_df.index, dtype=object))
umap_df["accepted_for_pseudo_label"] = accepted_series.astype(str).str.lower().map({"true": True, "false": False})

print("Loaded UMAP points from:", UMAP_POINTS_CSV)
print("Rows:", len(umap_df))
display(umap_df.head())
display(umap_df["split_label"].value_counts(dropna=False).rename("count").to_frame())

In [ ]:
UMAP_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def save_split_plot(umap_df: pd.DataFrame, figure_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 7))
    style_map = {
        "labeled_normal": dict(s=10, alpha=0.30, label="labeled_normal", color="#4d908e"),
        "labeled_defect": dict(s=12, alpha=0.50, label="labeled_defect", color="#f3722c"),
        "pseudo_unlabeled": dict(s=14, alpha=0.65, label="pseudo_unlabeled", color="#577590"),
    }

    for split_name, group in umap_df.groupby("split_label", sort=False):
        style = style_map.get(split_name, dict(s=12, alpha=0.50, label=str(split_name), color="#6b7280"))
        ax.scatter(group["umap_1"], group["umap_2"], **style)

    ax.set_title("10A-style UMAP of Seed07 Classifier Embeddings")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(frameon=False)
    plt.tight_layout()
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def save_score_plot(umap_df: pd.DataFrame, figure_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 7))
    score_mask = umap_df["split_label"] == "pseudo_unlabeled"
    scored_points = umap_df.loc[score_mask & umap_df["score"].notna()].copy()

    ax.scatter(
        umap_df.loc[~score_mask, "umap_1"],
        umap_df.loc[~score_mask, "umap_2"],
        s=8,
        alpha=0.12,
        color="#9ca3af",
        label="labeled reference",
    )

    if scored_points.empty:
        ax.scatter(
            umap_df.loc[score_mask, "umap_1"],
            umap_df.loc[score_mask, "umap_2"],
            s=18,
            alpha=0.65,
            color="#577590",
            label="pseudo_unlabeled",
        )
    else:
        sc = ax.scatter(
            scored_points["umap_1"],
            scored_points["umap_2"],
            c=scored_points["score"],
            cmap="viridis",
            s=18,
            alpha=0.85,
            label="pseudo_unlabeled",
        )
        fig.colorbar(sc, ax=ax, label="pseudo-label confidence")

    ax.set_title("10A-style UMAP Colored by Pseudo-Label Confidence")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(frameon=False)
    plt.tight_layout()
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def save_pseudo_label_plot(umap_df: pd.DataFrame, figure_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 7))
    score_mask = umap_df["split_label"] == "pseudo_unlabeled"

    ax.scatter(
        umap_df.loc[~score_mask, "umap_1"],
        umap_df.loc[~score_mask, "umap_2"],
        s=8,
        alpha=0.10,
        color="#d1d5db",
        label="labeled reference",
    )

    palette = plt.get_cmap("tab10")
    pseudo_points = umap_df.loc[score_mask].copy()
    pseudo_points = pseudo_points[pseudo_points["pseudo_label"].notna()]
    for idx, (pseudo_label, group) in enumerate(pseudo_points.groupby("pseudo_label", sort=True)):
        ax.scatter(
            group["umap_1"],
            group["umap_2"],
            s=18,
            alpha=0.70,
            label=str(pseudo_label),
            color=palette(idx % palette.N),
        )

    ax.set_title("10A-style UMAP with Pseudo Labels Highlighted")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

split_plot_path = UMAP_PLOTS_DIR / "umap_by_split_10a_style.notebook.png"
score_plot_path = UMAP_PLOTS_DIR / "umap_by_score_10a_style.notebook.png"
pseudo_label_plot_path = UMAP_PLOTS_DIR / "umap_by_pseudo_label_10a_style.notebook.png"

save_split_plot(umap_df=umap_df, figure_path=split_plot_path)
save_score_plot(umap_df=umap_df, figure_path=score_plot_path)
save_pseudo_label_plot(umap_df=umap_df, figure_path=pseudo_label_plot_path)

print("Saved:")
print(" -", split_plot_path)
print(" -", score_plot_path)
print(" -", pseudo_label_plot_path)